# 第7章 学習とは坂を下ること：微分と勾配降下法

目隠しで山を下りる方法と、PyTorchの自動微分

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：データから「ちょうどいい数」を自動で探す、いちばん小さな学習プログラム（勾配降下法）
- **身につく数学・理論**：モデル・パラメータ・損失の関係 ／ 二乗誤差 ／ 微分＝傾き（数値微分で確かめる） ／ 勾配降下法と学習率 ／ 偏微分と勾配（パラメータが2つ以上） ／ 連鎖律と誤差逆伝播法 ／ PyTorchの自動微分（`backward`）
- **新しく使うPython・PyTorch**：`requires_grad=True` ／ `loss.backward()` と `.grad` ／ `with torch.no_grad():` の本当の意味 ／ `.grad.zero_()`

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 90分 |
| 難易度 | ★★★☆☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> 「機械学習」の**学習**とは何をしているのか。その答えを、この章で手に入れます。キーワードは「坂を下る」。たった1つの数をデータに合わせる小さな問題から始めて、最後はPyTorchが自動で「下る方向」を計算してくれる仕組み（**自動微分**）まで進みます。これが、第8章以降のすべてのモデルの学習エンジンです。

> **🔗 これまでの復習とつながり：第3章・第5章・第6章とのつながり**
>
> - **第5章（損失）**：モデルの良し悪しを1つの数で表す成績表「損失（小さいほど良い）」を学びました。この章では、その損失を**自動で小さくしていく方法**を作ります。
> - **第6章（テンソルとベクトル）**：たくさんの数をひとまとめに扱うテンソルとベクトルを学びました。パラメータが複数になったときの傾き「勾配」は、まさにベクトルで表されます。
> - **第3章（`with torch.no_grad():`）**：文章生成のときに「おまじない」として使った `no_grad()` の本当の理由が、この章の後半でついに明かされます！

**ミナト**：微分……高校でやった気がするけど、記憶が霧の向こうです。

**ボンゴ先生**：安心してほしい。この章で使う微分は「坂の傾き」、それだけだ。公式を暗記する必要はない。傾きはコンピュータに測らせればいい。キミの仕事は「傾きが分かったら、どっちに進むか」を考えることだけだ。

## ボンゴ焼き問題

ボンゴ先生は、週末にたこ焼きならぬ「ボンゴ焼き」の屋台を出しています。焼いた時間と、売れた個数の記録がこちら。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
hours = [1, 2, 3, 4, 5]          # 焼いた時間
sold = [2.9, 6.2, 8.8, 12.1, 15.0]  # 売れた個数（10個単位）

だいたい「時間 × 3」くらいでしょうか。これを **モデル** にしてみます。

$$\text{予想} = w \times \text{時間}$$

$w$ は「1時間あたり何個売れるか」を表す数で、これが **パラメータ** です。$w$ をいくつにすれば、記録にいちばん合うでしょう？

まず「合っている度合い」を測るものさしを作ります。予想と実際の差を2乗して平均したものを **損失** にします。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def predict(w, x):
    return w * x

def loss_fn(w):
    total = 0.0
    for x, y in zip(hours, sold):
        total += (predict(w, x) - y) ** 2
    return total / len(hours)

for w in [1, 2, 3, 4]:
    print(f"w = {w} のとき、損失 = {loss_fn(w):.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
w = 1 のとき、損失 = 44.100
w = 2 のとき、損失 = 11.060
w = 3 のとき、損失 = 0.020
w = 4 のとき、損失 = 10.980
```

</details>

> **🐍 Python・PyTorchのポイント：** は2乗、-= は「引いて代入」**
>
> `(予想 - 実際) ** 2` の `**` は**べき乗**で、`a ** 2` は $a^2$（$a \times a$）です。第3章の `model(**inputs)` の `**`（辞書の中身を渡す書き方）とは、記号が同じだけで別物です。`**` の前に値があれば2乗、`(` のすぐ後にあれば辞書の受け渡し、と見分けます。<br>
> 第3章の `total += p`（足して代入）と同じ仲間に、`w -= a`（`w = w - a`、引いて代入）があります。この章の学習ループで使います。

> **💡 理論のポイント：二乗誤差**
>
> 「予想 − 実際」は、外れた方向によってプラスにもマイナスにもなります。そのまま平均すると打ち消し合ってしまうので、**2乗**してから平均します。これを **平均二乗誤差**（mean squared error、MSE）と呼びます。2乗するので、大きく外れたほど厳しく罰せられます。
>
> 第5章の言語モデルの損失（驚きの平均）とは式が違いますが、「小さいほど良い、モデルの成績表」という役割は同じです。

$w = 3$ あたりが良さそうです。グラフにしてみましょう。

📦 **準備セル**：ライブラリや教材ファイルを取得します。エラーが出た場合は `-q` を外して詳細を確認してください。

In [ ]:
!pip install -q matplotlib-fontja

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

ws = [i / 10 for i in range(0, 61)]
plt.figure(figsize=(6, 3.5))
plt.plot(ws, [loss_fn(w) for w in ws])
plt.xlabel("パラメータ w")
plt.ylabel("損失")
plt.title("損失の「谷」：いちばん低い場所の w が答え")
plt.grid(alpha=0.3)
plt.show()

![損失をwの関数として描いたグラフ。w=3付近を底とするお椀型](https://sakunokyumei.github.io/llm-research-textbook/img/figs/07-gradient/cell3_0.png)

お椀のような谷の形です。**学習とは、この谷の一番底を探すこと**です。

今回はパラメータが1個なので、グラフを描いて目で探せました。でも、第1章のLLMのパラメータは約5億個。5億次元のグラフは、描くのも見るのも無理です。そこで、グラフ全体を見ずに、**今いる場所の傾きだけを頼りに**下っていく方法を使います。

#### 📝 確認問題（`ch07-q01` · Lv.1 · 二乗誤差（MSE）で差を2乗する理由）

予想と実際の差 `pred - y` をそのまま平均するのではなく、**2乗してから平均する（平均二乗誤差 MSE）**最も大きな理由はどれですか？

- **a.** 差を2乗すると損失の値が必ずマイナスになって揃うため、グラフを描いたときに谷底が見やすくなるため
- **b.** 正負の外れが足し算で打ち消し合うのを防ぎ、大きな外れほど重く罰するため
- **c.** Pythonの仕様上、マイナスの小数が混ざった差のリストはそのままでは合計や平均を計算できないため
- **d.** 差を2乗して計算することで、最適なパラメータ $w$ が必ずきれいな整数値になるよう制限するため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：もし2点のデータで差が `+10` と `-10` だったとき、2乗せずにそのまま平均するとどうなるでしょう？
- **ヒント 2**：`(+10 + (-10)) / 2 = 0` と打ち消し合って大外れを見逃してしまいます。2乗すれば常に非負になり大外れほど重い罰になります。
- **正解**：`b`

**考え方**：差をそのまま平均すると `+10` と `-10` が打ち消し合って「誤差0（完璧！）」と誤判定してしまいます。2乗すればすべて 0 以上になり、しかも滑らかなお椀型になって微分もしやすくなります。

1. 外れ `+10` と `-10` をそのまま平均すると `0` になり、大外れを見逃す。
1. 2乗すると `100` と `100`、平均は `100` となり、外れの大きさが正しく反映される。
1. さらに誤差が2倍になると2乗誤差は4倍になるため、大きな外れを優先的に直そうとする。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q01

</details>

#### 📝 確認問題（`ch07-q02` · Lv.2 · 小さなデータでの平均二乗誤差（MSE）の手計算）

データが `hours = [1, 2]`、`sold = [3, 6]` の2点だけだとします。モデル $\text{予想} = w \times \text{時間}$ で $w = 2$ としたとき、平均二乗誤差（MSE）はいくつになりますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：平均二乗誤差（MSE）は、各データの `(予想 - 正解) ** 2` を計算してすべて足し、データ数で割ります。
- **ヒント 2**：$w=2$ のとき1点目の予想は `2`（正解 `3`）、2点目の予想は `4`（正解 `6`）です。差の2乗を足して `2` で割りましょう。
- **正解**：`2.5`

**考え方**：$\text{MSE} = \frac{1}{2} \left( (2 \times 1 - 3)^2 + (2 \times 2 - 6)^2 \right)$ を順に計算します。

1. 1点目（$x=1, y=3$）：予想 $2 \times 1 = 2$、差 $2 - 3 = -1$、2乗 $(-1)^2 = 1$。
1. 2点目（$x=2, y=6$）：予想 $2 \times 2 = 4$、差 $4 - 6 = -2$、2乗 $(-2)^2 = 4$。
1. 平均：$(1 + 4) / 2 = 2.5$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q02

</details>

*(この節の追加問題 `ch07-q03` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html で自動採点・復習できます)*

## 目隠しで山を下りる

**ボンゴ先生**：キミは濃い霧の山の中にいる。目隠しをされていて、遠くは見えない。分かるのは**足元の傾き**だけ。どうやって谷底まで下りる？

**ミナト**：足元が下っている方向に、一歩ずつ進めばいいんじゃないですか？

**ボンゴ先生**：正解！ それが **勾配降下法**（こうばいこうかほう）だ。人類の機械学習の大部分は、この「足元の傾きを見て一歩下る」でできている。拍子抜けするほど単純だろう？

足元の傾きを測ってみましょう。$w$ をほんの少し（$h = 0.0001$）だけ動かして、損失がどれだけ変わるかを見ます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def slope(w, h=0.0001):
    return (loss_fn(w + h) - loss_fn(w)) / h

for w in [1, 2, 3, 4]:
    print(f"w = {w} での傾き：{slope(w):8.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
w = 1 での傾き： -44.039
w = 2 での傾き： -22.039
w = 3 での傾き：  -0.039
w = 4 での傾き：  21.961
```

</details>

#### 📐 数学のポイント：微分 $\frac{dL}{dw}$：9ステップで理由から理解する

**① 解決したい問題**：グラフ全体を見ずに、「いまのパラメータ $w$ をほんの少し右に動かしたら、損失 $L$ が増えるか減るか（どれくらい急な坂か）」を1つの数で知りたい。

**② そのままでは困る理由**：損失 $L(w)$ の値だけ見ても（たとえば「いま損失は 11.06」）、右に行けば下がるのか左に行けば下がるのか分かりません。

**③ 新しい発想**：$w$ をごく小さな幅 $h$ だけ右へ動かしたときの **「縦の変化 ÷ 横の変化」**（＝坂の傾き）を測る。
$$\frac{L(w + h) - L(w)}{h}$$
$h$ を限りなく $0$ に近づけた正確な傾きを **微分**（derivative）と言い、$\dfrac{dL}{dw}$ と書きます。

**④ 記号の読み方テーブル**：

| 記号 | 読み方 | 意味 |
|---|---|---|
| $d$ | ディー（differential） | 「ほんの少しの変化」を表す印（掛け算の文字ではないので約分不可！） |
| $\dfrac{dL}{dw}$ | エルをダブリューで微分したもの | $w$ をほんの少し増やしたときの「$L$ の変化 ÷ $w$ の変化」（坂の傾き） |
| $h$ | エイチ | 数値微分で使う微小なずらし幅（例：`0.0001`） |

**⑤ 小さな数字で手計算**：シンプルな谷 $L(w) = (w - 3)^2$（谷底は $w = 3$）で、$h = 0.1$ として傾きを測ってみます。
- $w = 1$：$L(1) = (-2)^2 = 4$、$L(1.1) = (-1.9)^2 = 3.61$ → 傾き $= (3.61 - 4) / 0.1 = -3.9$（**マイナス＝右へ行くと下る**）
- $w = 4$：$L(4) = 1^2 = 1$、$L(4.1) = 1.1^2 = 1.21$ → 傾き $= (1.21 - 1) / 0.1 = +2.1$（**プラス＝右へ行くと上る → 左へ下る**）
- $w = 3$：$L(3) = 0$、$L(3.1) = 0.01$ → 傾き $= 0.1 \approx 0$（**ほぼ 0 ＝谷底**）

**⑥ 符号のまとめ（超重要）**：
- $\dfrac{dL}{dw} < 0$（マイナス）→ 右（$w$ を大きくする方向）が下り坂
- $\dfrac{dL}{dw} > 0$（プラス）→ 左（$w$ を小さくする方向）が下り坂
- $\dfrac{dL}{dw} = 0$ → 平ら（谷底に到達）

つまり、**傾きと逆の方向に進めば必ず下れる**のです。上のコードのように微小な $h$ で実際に測る方法を **数値微分** と言います。

#### 📝 確認問題（`ch07-q04` · Lv.1 · 微分 dL/dw の読み方と数値微分の意味）

数式 $\dfrac{dL}{dw}$ の読み方と意味として**正しい**ものはどれですか？

- **a.** お椀型グラフの谷底の高さである「損失 $L$ の最小値そのもの」をそのまま表す記号である
- **b.** 変数 $d$ が分子と分母の両方に掛かっている分数なので、$d$ 同士を約分して $L / w$ とまったく同じ値になる
- **c.** 「$L$ を $w$ で微分したもの」。$w$ を少し動かしたときの比（グラフの傾き）を表す
- **d.** ハイパーパラメータの学習率 $\eta$ を学習データの総件数で割った平均の歩幅を表す記号である

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：記号 $d$ は掛け算の文字ではなく、英語の difference（ごく小さな変化）に由来します。
- **ヒント 2**：縦の小さな変化 $dL$ を横の小さな変化 $dw$ で割った比、すなわちその点における坂の傾きを表します。
- **正解**：`c`

**考え方**：微分 $\frac{dL}{dw}$ は「パラメータ $w$ をごくわずかに増やしたとき、損失 $L$ がどれだけの比率で増減するか（坂の傾き）」を表します。

1. $w$ を $h$ だけ右へ動かすと、損失は $L(w + h) - L(w)$ だけ変化する。
1. その比 $\frac{L(w+h) - L(w)}{h}$ が数値微分（近似の傾き）。
1. $h$ を極限まで 0 に近づけた正確な傾きを $\frac{dL}{dw}$ と書く。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q04

</details>

#### 📝 確認問題（`ch07-q05` · Lv.2 · 数値微分 (f(w+h)-f(w))/h の手計算）

損失関数が $L(w) = w^2$ のとき、$w = 3$、$h = 0.1$ で数値微分 $\dfrac{L(w + h) - L(w)}{h}$ を手計算してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：数値微分の式 $\dfrac{L(w + h) - L(w)}{h}$ に $L(w) = w^2, w = 3, h = 0.1$ を当てはめます。
- **ヒント 2**：$L(3.1) = 3.1^2 = 9.61$、$L(3) = 9.00$ です。差 `9.61 - 9.00` を `0.1` で割りましょう。
- **正解**：`6.1`

**考え方**：正確な微分は $2w = 2 \times 3 = 6$ ですが、$h = 0.1$ だけ右にずらして測るため `6.1` と少しだけずれます（$h$ を小さくするほど `6.0` に近づきます）。

1. $L(3 + 0.1) = 3.1^2 = 9.61$。
1. $L(3) = 3^2 = 9.00$。
1. $(9.61 - 9.00) / 0.1 = 0.61 / 0.1 = 6.1$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q05

</details>

*(この節の追加問題 `ch07-q06` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html で自動採点・復習できます)*

## 勾配降下法

一歩の大きさを **学習率**（learning rate）と呼び、数式ではギリシャ文字の **$\eta$（イータ）**、コードでは `lr` と書きます。ルールはこれだけ。

$$w \leftarrow w - \eta \frac{dL}{dw} \qquad \left(\text{コード：}\texttt{w = w - lr * slope(w)}\right)$$

#### 📐 数学のポイント：勾配降下法：なぜ「引き算（$-$）」で両方うまくいくのか？

**① 解決したい問題**：いま測った傾き $\dfrac{dL}{dw}$ を使って、パラメータ $w$ を谷底の方向へ1歩だけ動かしたい。

**② 記号の読み方**：
- $\leftarrow$（左向き矢印）：「右辺の計算結果で左辺の変数を**更新（上書き）する**」という意味（プログラミングの `=` と同じ）
- $\eta$（イータ）：**学習率**（learning rate、`lr`）。一歩の歩幅を決める正の数（例：`0.01`）

**③ なぜ足し算ではなく「引き算」なのか？（符号の完全分析）**：

| いまの場所 | 傾き $\dfrac{dL}{dw}$ の符号 | $-\eta \dfrac{dL}{dw}$ の符号 | $w$ の動き | 結果 |
|---|---|---|---|---|
| 谷底の**左側**（例：$w=1$） | **マイナス**（例：$-4$） | マイナス × マイナス ＝ **プラス** | 右（大きい側）へ進む | 谷底へ下る！ |
| 谷底の**右側**（例：$w=5$） | **プラス**（例：$+4$） | マイナス × プラス ＝ **マイナス** | 左（小さい側）へ進む | 谷底へ下る！ |
| ちょうど**谷底**（$w=3$） | **$0$** | **$0$** | 動かない | 谷底で止まる！ |

**④ 小さな数字で2ステップ手計算**：谷 $L(w) = (w - 3)^2$（正確な傾きは $\frac{dL}{dw} = 2(w - 3)$）を、$w = 1.0$、学習率 $\eta = 0.2$ からスタートしてみます。
- **ステップ1**：傾き $= 2(1.0 - 3) = -4.0$ → $w \leftarrow 1.0 - 0.2 \times (-4.0) = 1.0 + 0.8 = 1.8$（損失は $4.0 \to 1.44$ に激減！）
- **ステップ2**：傾き $= 2(1.8 - 3) = -2.4$ → $w \leftarrow 1.8 - 0.2 \times (-2.4) = 1.8 + 0.48 = 2.28$（損失は $1.44 \to 0.518$ にさらに減少！）

坂が急なうちは大きく（`+0.8`）、谷底に近づくと傾きが緩やかになって自然に小さく（`+0.48`）進んでいるのがわかります。

```python no-run
# 上の手計算2ステップを最小Pythonで確認
w, lr = 1.0, 0.2
for step in range(2):
    grad = 2 * (w - 3)          # L(w) = (w - 3)^2 の正確な傾き
    w = w - lr * grad           # 傾きと逆向きに一歩進む
    print(f"step {step + 1}: w = {w:.2f}, loss = {(w - 3)**2:.3f}")
# step 1: w = 1.80, loss = 1.440
# step 2: w = 2.28, loss = 0.518
```

「傾きと逆向きに、傾きの大きさに比例して進む」。ボンゴ焼きのデータで実際に15ステップ下ってみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
w = 0.0
lr = 0.01
for step in range(15):
    w = w - lr * slope(w)
    if step % 2 == 0:
        print(f"ステップ {step:2d}：w = {w:.4f}  損失 = {loss_fn(w):.4f}")

> **🐍 Python・PyTorchのポイント：% は割り算の余り**
>
> `step % 2` は「`step` を2で割った余り」です。偶数なら0、奇数なら1なので、`if step % 2 == 0:` は「偶数のステップのときだけ表示する」という意味になります。この先の学習ループでは、`if step % 500 == 0:`（500ステップごとに表示）の形で何度も出てきます。

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ  0：w = 0.6604  損失 = 60.3252
ステップ  2：w = 1.5773  損失 = 22.3426
ステップ  4：w = 2.1351  損失 = 8.2831
ステップ  6：w = 2.4745  損失 = 3.0788
ステップ  8：w = 2.6810  損失 = 1.1523
ステップ 10：w = 2.8066  損失 = 0.4392
ステップ 12：w = 2.8830  損失 = 0.1752
ステップ 14：w = 2.9295  損失 = 0.0775
```

</details>

$w = 0$ から出発して、15歩で3の近くまで来ました。誰も「答えは3だよ」と教えていないのに、傾きだけを頼りに谷底へ向かっています。最初は坂が急なので大きく進み、谷底に近づくと坂が緩やかになって、歩幅が自然に小さくなっているのにも注目してください。

*(対話デモはWeb教科書ページで確認できます：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html)*

> **🔬 やってみよう：学習率を変えてみよう**
>
> 上のデモで、学習率をいろいろ変えてみましょう。
>
> - **小さすぎる**（0.05など）：確実だけど、なかなか谷底に着かない。カタツムリ
> - **ちょうどいい**（0.3など）：すいすい下る
> - **大きすぎる**（2近く）：谷を飛び越えて、反対側の斜面に着地。行ったり来たりして、最悪どこかへ飛んでいく
>
> ただし、「ちょうどいい学習率」は坂の急さで変わります。デモの坂はゆるやかなので0.3でちょうどよく、ボンゴ焼きの坂はずっと急なので、0.01〜0.05くらいがちょうどいい値です。ボンゴ焼きのコードで `lr = 0.1` にして試してみてください。デモでは小さめの0.1でも、ボンゴ焼きでは損失が爆発するのが見られます。学習率は、機械学習でいちばん大事なハイパーパラメータです。モデルやデータが変わったら、また探し直す必要があります。
>
> もう1つ、デモで「別の場所から」を何回か押してみてください。デモの坂には、本当の谷底のほかに、途中に**小さなくぼみ**があります。スタート地点によっては、そこで止まってしまいます。勾配降下法は「今いる場所の傾き」しか見ないので、もっと深い谷が別の場所にあっても気づけないのです（**局所最適解**と言います）。目隠しで山を下りるのですから、仕方ありません。ただ、パラメータが何億個もある実際のLLMでは、こうした浅いくぼみに完全にはまって動けなくなることは、意外と少ないと考えられています。

#### 📝 確認問題（`ch07-q07` · Lv.2 · 勾配降下法の1ステップ更新 w ← w - η (dL/dw) の手計算）

現在 $w = 5.0$ で、損失の $w$ についての傾きが $\dfrac{dL}{dw} = +5$ でした。学習率 $\eta = 0.1$ の勾配降下法を1ステップ行うと、新しい $w$ はいくつになりますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：勾配降下法の1ステップ更新式 $w_{\text{新}} = w - \eta \cdot \frac{dL}{dw}$ を使います。
- **ヒント 2**：移動量 $\eta \times \frac{dL}{dw} = 0.1 \times 5$ を現在の値 $w = 5.0$ から引きましょう。
- **正解**：`4.5`

**考え方**：傾きが $+5$（右に行くと上る）なので、$0.1 \times 5 = 0.5$ だけ左（小さい側）へ進みます。

1. 現在の値：$w = 5.0$、傾き $= +5$、学習率 $\eta = 0.1$。
1. 移動量：$\eta \times \frac{dL}{dw} = 0.1 \times 5 = 0.5$。
1. 更新：$w_{\text{新}} = 5.0 - 0.5 = 4.5$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q07

</details>

#### 📝 確認問題（`ch07-q08` · Lv.2 · 学習率 η が大きすぎる・小さすぎるときの挙動）

勾配降下法で**学習率 $\eta$（`lr`）を大きくしすぎる**と、どのような現象が起きますか？

- **a.** 傾きの大きさにかかわらず、常に最初の1ステップだけで誤差ゼロの谷底へぴったり着地できる
- **b.** 学習率の上限を超えると傾きの計算自体がスキップされ、パラメータが初期値のまま動かなくなる
- **c.** 一歩が大きすぎて谷底を飛び越え、反対側の斜面を行き来したり発散したりする
- **d.** 一歩が小さくなりすぎてしまい、何千ステップ回しても谷底にほとんど近づかないまま止まってしまう

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：U字型の谷を下りるとき、1歩の歩幅が谷の幅より大きすぎたらどこに着地するか想像しましょう。
- **ヒント 2**：谷底を飛び越えて反対側のさらに高い斜面へ跳ね返り、振動や発散（`inf`/`nan`）が起きます。
- **正解**：`c`

**考え方**：学習率は機械学習で最も重要なハイパーパラメータです。坂が急なモデルほど小さめの学習率が必要になります。

1. 小さすぎる：安全だがカタツムリのように遅い。
1. ちょうどよい：最初は大きく、谷底に近づくにつれ傾きが緩やかになり自然と小幅に着地する。
1. 大きすぎる：谷底を飛び越えて振動・発散（損失が `inf` や `nan` に爆発）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q08

</details>

*(この節の追加問題 `ch07-q09` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html で自動採点・復習できます)*

## パラメータが2つになったら：勾配

実際には、時間がゼロでも少しは売れる（常連さんがいる）かもしれません。モデルを少し賢くします。

$$\text{予想} = w \times \text{時間} + b$$

パラメータが $w$ と $b$ の2つになりました。今度は2次元の地形（山と谷）を下ることになります。傾きも「$w$ 方向の傾き」と「$b$ 方向の傾き」の2つを測ります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def loss2(w, b):
    return sum((w * x + b - y) ** 2 for x, y in zip(hours, sold)) / len(hours)

def slopes(w, b, h=0.0001):
    dw = (loss2(w + h, b) - loss2(w, b)) / h   # w だけ少し動かす
    db = (loss2(w, b + h) - loss2(w, b)) / h   # b だけ少し動かす
    return dw, db

w, b = 0.0, 0.0
for step in range(2000):
    dw, db = slopes(w, b)
    w, b = w - 0.02 * dw, b - 0.02 * db
print(f"w = {w:.3f}, b = {b:.3f}, 損失 = {loss2(w, b):.4f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
w = 3.010, b = -0.029, 損失 = 0.0198
```

</details>

#### 📐 数学のポイント：偏微分 $\frac{\partial L}{\partial w}$ と勾配 $\nabla L$：多変数でもやり方は同じ

**① 解決したい問題**：パラメータが2個（$w$ と $b$）、さらには5億個になったとき、それぞれのつまみをどちらへどれだけ動かせばよいかを表したい。

**② 新しい発想**：「他のパラメータは全部ピタッと止めたまま、**1つだけ**ほんの少し動かしたときの傾き」を測る。これを **偏微分**（へんびぶん、partial derivative）と言い、丸っこい **$\partial$** を使って書きます。そして全パラメータの偏微分を1つのベクトルに並べたものが **勾配**（gradient）です。

$$\nabla L = \left( \frac{\partial L}{\partial w}, \frac{\partial L}{\partial b} \right), \qquad w \leftarrow w - \eta \frac{\partial L}{\partial w}, \quad b \leftarrow b - \eta \frac{\partial L}{\partial b}$$

**③ 記号の読み方テーブル**：

| 記号 | 読み方 | 意味 |
|---|---|---|
| $\partial$ | ラウンドディー／デル／パーシャル | 「他の変数は固定して、これだけ少し動かす」という偏微分の合図 |
| $\dfrac{\partial L}{\partial w}$ | エルをダブリューで偏微分したもの | $b$ を止めて $w$ だけ少し動かしたときの $L$ の傾き |
| $\nabla$ | ナブラ（nabla） | すべてのパラメータの偏微分を並べてベクトルにする演算子 |
| $\nabla L$ | 勾配（グラディエント） | 「最も急に**上る**方向」を指す矢印ベクトル（逆向き $-\nabla L$ が最も急に下る方向！） |

**④ 小さな数字で手計算**：データ1点（$x=2, y=7$）の損失 $L(w, b) = (2w + b - 7)^2$ を $(w, b) = (2, 1)$ で偏微分してみます（このとき $2w + b - 7 = 4 + 1 - 7 = -2$）。
- **$w$ で偏微分**（$b=1$ を定数扱い → $L = (2w - 6)^2$）：$\dfrac{\partial L}{\partial w} = 2(2w - 6) \cdot 2 = 4(-2) = -8$
- **$b$ で偏微分**（$w=2$ を定数扱い → $L = (b - 3)^2$）：$\dfrac{\partial L}{\partial b} = 2(b - 3) \cdot 1 = 2(-2) = -4$
- よって勾配ベクトルは $\nabla L = (-8, -4)$。学習率 $\eta = 0.1$ なら $(w, b) \leftarrow (2 - 0.1(-8), \; 1 - 0.1(-4)) = (2.8, 1.4)$ と更新され、$2(2.8) + 1.4 = 7.0$ でなんと1歩で損失 `0` に着地します！

パラメータが5億個でも、勾配が5億個の数の並び（ベクトル）になるだけで、やることはまったく同じです。……ただし、数値微分で5億個の偏微分を測るには、モデルを5億回動かす必要があります。それはさすがに無理。そこで次の秘密兵器の出番です。

#### 📝 確認問題（`ch07-q10` · Lv.2 · 偏微分 ∂L/∂w と勾配ベクトル ∇L の読み方・意味）

パラメータが $w$ と $b$ の2つあるときの偏微分 $\dfrac{\partial L}{\partial w}$ と、勾配 $\nabla L = \left(\dfrac{\partial L}{\partial w}, \dfrac{\partial L}{\partial b}\right)$ についての正しい説明はどれですか？

- **a.** 偏微分と勾配ベクトルは2変数のとき専用の記号であり、パラメータが3個以上になると定義できない
- **b.** $\partial$ は他を固定し $w$ だけ動かす傾き、$\nabla L$ は全偏微分を並べた最急上昇ベクトル
- **c.** $\partial$ は掛け算記号、$\nabla L$ は最も急に**下る**向きを指す1個だけのスカラー値（傾きの平均）
- **d.** $\frac{\partial L}{\partial w}$ は $w$ と $b$ の2つを同時に2倍へ動かしたときの損失の合計変化量を表す

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「偏（かたよ）って微分する」＝他のつまみを固定して1つだけ動かす、そして勾配 $\nabla L$ が「上り」と「下り」のどちらを指すか確認しましょう。
- **ヒント 2**：$\nabla L$ は全パラメータの偏微分を並べたベクトルで最急上昇方向を指します（だからマイナスを付けて下ります）。
- **正解**：`b`

**考え方**：複数のパラメータがある場合も、「1つずつ固定して傾きを測り、全部を並べたベクトル $\nabla L$ の逆向きへ進む」だけで多次元の谷を下れます。

1. $\frac{\partial L}{\partial w}$：$b$ を止めて $w$ だけ $+h$ 動かしたときの傾き。
1. $\frac{\partial L}{\partial b}$：$w$ を止めて $b$ だけ $+h$ 動かしたときの傾き。
1. $\nabla L = \left(\frac{\partial L}{\partial w}, \frac{\partial L}{\partial b}\right)$ は最急上昇方向。$-\eta \nabla L$ が最急降下の一歩。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q10

</details>

#### 📝 確認問題（`ch07-q11` · Lv.3 · 巨大LLMで数値微分が使えない理由）

5億個のパラメータを持つLLMを学習させるとき、数値微分ではなく**自動微分（誤差逆伝播法）**が必須になる決定的な理由はどれですか？

- **a.** 数値微分は必ず符号が逆の傾きを出してしまうため、小さなモデルであっても勾配降下法で損失を下げることができないから
- **b.** 数値微分は1歩に約5億回の順伝播が必要だが、逆伝播なら順伝播1回＋逆伝播1回で全勾配が求まるから
- **c.** GPUは足し算と掛け算専用であり、数値微分に必要な引き算と割り算をハードウェア上で一切実行できないから
- **d.** 数値微分はPythonの仕様上1変数の関数にしか書けず、2個以上のパラメータを持つモデルではそもそも実装できないから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：パラメータ数 $P$ が5億個のとき、数値微分と誤差逆伝播法で「順伝播を何回行う必要があるか」を比べましょう。
- **ヒント 2**：数値微分は $P$ に比例（約5億回）、誤差逆伝播法は $P$ によらず定数回（順伝播1回＋逆伝播1回）で全勾配が揃います。
- **正解**：`b`

**考え方**：パラメータ数を $P$ とすると、数値微分は1ステップに $P+1$ 回の順伝播が必要ですが、誤差逆伝播法は $P$ の大きさによらず「順伝播1回＋逆伝播1回」で完了します。

1. 数値微分：5億パラメータ → 1ステップごとに順伝播を約5億回（何年もかかる）。
1. 誤差逆伝播法：計算グラフを逆向きに1回たどるだけ → 全5億個の `.grad` が1回で揃う。
1. さらに微小な $h$ による近似誤差もなく正確。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q11

</details>

## PyTorchの自動微分

PyTorchは、計算の過程を記録しておいて、**正確な勾配を一発で計算**してくれます。これを **自動微分**（autograd）と言います。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import torch

x = torch.tensor(hours, dtype=torch.float32)
y = torch.tensor(sold)

w = torch.tensor(1.0, requires_grad=True)   # 「この数の勾配を知りたい」と宣言
loss = ((w * x - y) ** 2).mean()
loss.backward()                              # 勾配を計算！
print("損失：", loss.item())
print("自動微分の傾き：", w.grad.item())
print("数値微分の傾き：", slope(1.0))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
損失： 44.099998474121094
自動微分の傾き： -44.040000915527344
数値微分の傾き： -44.03890000006072
```

</details>

`loss.backward()` を呼ぶだけで、`w.grad` に傾き $\dfrac{dL}{dw}$ が入りました。数値微分とほぼ同じ値ですが、小数点以下3桁目からずれています。数値微分は「少し右に動かした」ときの傾きなので誤差があり、自動微分のほうが正確な値です。

では、自動微分で学習ループを書き直しましょう。これが、この先何度も書くことになる **学習ループの基本形** です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.02

for step in range(2000):
    loss = ((w * x + b - y) ** 2).mean()   # ① 予想して、損失を計算
    loss.backward()                         # ② 勾配を計算
    with torch.no_grad():                   # ③ パラメータを更新（ここは記録しない）
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()                          # ④ 勾配を0に戻す
    b.grad.zero_()

print(f"w = {w.item():.3f}, b = {b.item():.3f}, 損失 = {loss.item():.4f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
w = 3.010, b = -0.030, 損失 = 0.0198
```

</details>

> **⚠️ つまずき注意：④を忘れると大変**
>
> PyTorchの `.grad` は、`backward()` を呼ぶたびに **足し算されていく**（累積する）仕様です。④の `zero_()` を忘れると、前のステップの勾配がどんどん雪だるま式に積み上がり、学習がめちゃくちゃになります。エラーメッセージが出ずに結果だけがおかしくなるので、気づきにくいバグの代表格です。学習の様子がおかしいときは、まずここを疑いましょう。
>
> なぜ自動で0にリセットしてくれないのかというと、GPUメモリが足りないときに何回かに分けて勾配を足し合わせる技（第16章の「勾配累積」）などで、この累積仕様が役に立つからです。便利な機能の裏返しとして、普段は毎回手動でリセット（ゼロクリア）してあげる必要があります。

> **💡 理論のポイント：第3章の `no_grad` の謎が解ける**
>
> ③で `with torch.no_grad():` を使っているのは、「パラメータの更新」は学習のための計算ではないので、記録の外でやる必要があるからです。記録したまま `w -= lr * w.grad` を実行すると、PyTorchは「記録中のパラメータを直接書き換えようとしている」と判断して、`RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.` というエラーで止めます。
>
> そして第3章では、文章を生成するだけ（学習しない）だったので、`no_grad` で記録を止めて、メモリと時間を節約していたのでした。あのおまじないの意味が、ようやく分かりましたね。

<details><summary><b>基礎の補足：PyTorch 学習ループ4ステップの解剖（なぜ `zero_()` と `no_grad()` が必須なのか）</b>（クリックで開く）</summary>

| ステップ | コード | 役割 | 忘れるとどうなる？ |
|---|---|---|---|
| ① 順伝播＋損失 | `loss = ...` | 予想と損失を計算しつつ、計算グラフを記録 | `no_grad()` の中に入れると `backward()` でエラー |
| ② 逆伝播 | `loss.backward()` | 計算グラフを逆向きにたどり `.grad` を算出 | `.grad` が `None` のままで更新できない |
| ③ 更新 | `with torch.no_grad(): w -= lr * w.grad` | 記録を切って、勾配の逆向きへ一歩下る | `no_grad()` を忘れると `RuntimeError`（leaf in-place error） |
| ④ 勾配リセット | `w.grad.zero_()` | 次のステップに古い勾配が足されないよう0に戻す | エラーは出ないが勾配が雪だるま式に足され学習が崩壊！ |

```python no-run
# なぜ zero_() を忘れると危険なのか？（最小の再現例）
t = torch.tensor(2.0, requires_grad=True)
(3 * t).backward()          # 1回目：3t の微分は 3.0
print(t.grad.item())        # → 3.0
(3 * t).backward()          # zero_() せずに2回目！
print(t.grad.item())        # → 6.0（上書きではなく 3.0 + 3.0 に足し算されてしまう！）
```

あわせて、自動微分の記録を止める・切り離す3つの道具の使い分けも押さえておきましょう（第11章以降で頻出します）。

| 書き方 | 対象と返り値 | 何をするか | 使いどころ |
|---|---|---|---|
| `with torch.no_grad():` | ブロック全体 | 中の計算グラフ記録をすべてオフにする | パラメータ更新・検証・生成（推論） |
| `t.detach()` | テンソル1個 → 記録なしの `Tensor` | 値（shape・dtype・device）はそのままに計算グラフだけ切り離す | グラフ描画・Attention重み観察・別計算への受け渡し |
| `s.item()` | 要素1個のテンソル → Pythonの `float`/`int` | 数値1個を取り出す（記録も切れる） | `losses.append(loss.item())` のようにログ記録（テンソルのまま溜めるとメモリリーク！） |

</details>

#### 📝 確認問題（`ch07-q12` · Lv.2 · w.grad.zero_() を忘れると何が起きるか）

PyTorchの学習ループで `w.grad.zero_()`（勾配のリセット）を**書き忘れる**と、何が起きますか？

- **a.** PyTorchは毎回の `backward()` の直前に自動で `.grad` を0へ戻してくれるため、書き忘れても問題は起きない
- **b.** 勾配ではなくパラメータ `w` 自身の値が自動的に `0` にリセットされ、学習が毎回最初からやり直しになる
- **c.** 2回目の `backward()` を呼んだ瞬間に `ZeroDivisionError` が発生し、学習ループが強制終了する
- **d.** エラーは出ないが、新しい勾配が古い `.grad` に足し算（累積）され続け更新が壊れる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：PyTorchの `.grad` は `backward()` のたびに「上書き」されるのか「足し算（累積）」されるのか思い出しましょう。
- **ヒント 2**：エラーを出さずに古い勾配へ新しい勾配が `+=` で足され続けるため、気づきにくいバグになります。
- **正解**：`d`

**考え方**：PyTorchが勾配を累積する仕様なのは、第16章の「勾配累積（小さいバッチを何回か足して大きいバッチ相当にする技）」で活用できるからです。

1. 1回目の `backward()`：`w.grad` が `-44` になる。
1. `zero_()` を忘れたまま2回目の `backward()`（本来の勾配 `-30`）：`w.grad` が `-44 + (-30) = -74` になってしまう！
1. エラーが出ないまま暴走するため、毎ステップ必ずリセット（`zero_()` や `= None`）する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q12

</details>

#### 📝 確認問題（`ch07-q13` · Lv.2 · with torch.no_grad(): が更新時と推論時の両方で必要な理由）

学習ループのパラメータ更新で `with torch.no_grad():` の中に `w -= lr * w.grad` を入れる理由はどれですか？

- **a.** `with torch.no_grad():` を付けずに更新すると、計算済みの `w.grad` の値が強制的に `0` に消えてしまうから
- **b.** パラメータの書き換え処理だけをGPUではなく強制的にCPU上へ移して安全に実行させるため
- **c.** `with torch.no_grad():` の中に書くことで、PyTorchが勾配の大きさに応じて学習率を自動調整してくれるから
- **d.** 更新は微分グラフに含めるべきでなく、記録ONのまま `w -= ...` するとエラーになるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`requires_grad=True` のパラメータを、自動微分の記録がONのまま `-=`（インプレース書き換え）するとどうなるでしょう？
- **ヒント 2**：パラメータ更新は微分したい計算ではないため、`torch.no_grad()` で記録を一時停止してから安全に書き換えます。
- **正解**：`d`

**考え方**：順伝播（損失計算）は記録ON、パラメータ更新と推論（第3章）は記録OFF（`torch.no_grad()`）と役割がはっきり分かれています。

1. 損失計算：後で `backward()` したいので計算グラフを記録する。
1. パラメータ更新：記録したまま `w -= ...` すると `RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.` になる。
1. `with torch.no_grad():` で記録を一時オフにして安全に値を書き換える。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q13

</details>

## 自動微分の中身：連鎖律

自動微分はどうやって勾配を計算しているのでしょう？ 鍵は **連鎖律**（れんさりつ、chain rule）です。

#### 📐 数学のポイント：連鎖律 $\frac{dy}{dx} = \frac{dy}{du} \frac{du}{dx}$ と計算グラフの手計算トレース

**① 解決したい問題**：何段も計算を重ねた式（ニューラルネットワークは何十段もの計算の合成！）について、一番最後の出力（損失）を一番最初の入力やパラメータで微分したい。

**② 新しい発想**：計算を途中の変数 $u$ に分解し、**各ステップの局所的な傾きを掛け算でつなぐ**（伝言ゲームの倍率の掛け算）。

$$u = 3x + 1, \qquad y = u^2 \qquad \Longrightarrow \qquad \frac{dy}{dx} = \frac{dy}{du} \times \frac{du}{dx} = 2u \times 3$$

**③ 具体的な数字で「順伝播 → 逆伝播」を手計算トレース**（$x = 2$ のとき）：

1. **順伝播（左から右へ値を計算し、途中結果 $u$ をメモする）**：
   - 入力：$x = 2$
   - 中間：$u = 3 \times 2 + 1 = 7$（← **この $u = 7$ を覚えておく！**）
   - 出力：$y = 7^2 = 49$
2. **逆伝播（右から左へ、傾きを掛け算しながら戻す）**：
   - ゴール自身：$\dfrac{dy}{dy} = 1$（$y$ が1動けば $y$ は1動く）
   - 2乗を逆走：$\dfrac{dy}{du} = 1 \times 2u = 1 \times (2 \times 7) = 14$（メモしておいた $u = 7$ を使う！）
   - $3x + 1$ を逆走：$\dfrac{dy}{dx} = \dfrac{dy}{du} \times \dfrac{du}{dx} = 14 \times 3 = 42$

```python no-run
# 順伝播＋手動逆伝播（PyTorch の autograd が裏でやっていることそのもの）
x = 2.0
u = 3 * x + 1           # 順伝播①：u = 7.0 をメモ
y = u ** 2              # 順伝播②：y = 49.0
dy_dy = 1.0             # 逆伝播の出発点
dy_du = dy_dy * (2 * u) # 1.0 * 14.0 = 14.0
dy_dx = dy_du * 3.0     # 14.0 * 3.0 = 42.0
print(f"y={y}, dy/du={dy_du}, dy/dx={dy_dx}")
```

**途中の傾きを掛け算でつないでいく**。これが連鎖律です。伝言ゲームで、各人が「受け取った話を3倍に盛って伝える」「14倍に盛って伝える」をすると、最後は $3 \times 14 = 42$ 倍に盛られて届くのと同じです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
x2 = torch.tensor(2.0, requires_grad=True)   # 上のデータの x と区別するため、x2 という名前に
u = 3 * x2 + 1
y2 = u ** 2
y2.backward()
print("自動微分：", x2.grad.item())
print("連鎖律で手計算：", 2 * u.item() * 3)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
自動微分： 42.0
連鎖律で手計算： 42.0
```

</details>

> **💡 理論のポイント：誤差逆伝播法（バックプロパゲーション）**
>
> PyTorchは、計算するたびに「何と何から、どんな計算で作られたか」を記録しています（**計算グラフ**）。`backward()` を呼ぶと、損失から出発して計算を**逆向きに**たどりながら、連鎖律で傾きを掛け算していきます。
>
> <div class="math-display">順伝播：x (=2) →(×3+1)→ u (=7) →(2乗)→ y (=49)<br>逆伝播：dy/dx (=42) ←(×3)← dy/du (=14) ←(×2u)← dy/dy (=1)</div>
>
> 5億個のパラメータがあっても、この逆向きの伝言ゲームを**1回**やるだけで、全部の勾配がまとめて求まります。この方法を **誤差逆伝播法**（ごさぎゃくでんぱほう、backpropagation）と言います。ニューラルネットワークの学習を現実的なものにした、最重要の発明です。

> **📘 基本情報とのつながり：誤差逆伝播法**
>
> 基本情報技術者試験のシラバスにも、AIの用語として「誤差逆伝播法」「勾配降下法」「学習率」が登場します。試験では用語の意味を問われるだけですが、あなたはもう**自分で動かしたことがある**人です。暗記カードより、ボンゴ焼きのほうが忘れにくいはずです。

<details><summary><b>もっと深く：微分の公式を少しだけ</b>（クリックで開く）</summary>

この教科書では、微分はPyTorchに任せます。でも、いくつかの公式を知っていると、式を読むときに便利です。

| 関数 | 微分 | ひとこと |
|---|---|---|
| $x^2$ | $2x$ | 2乗の傾きは2倍の $x$ |
| $x^n$ | $n x^{n-1}$ | 上の一般形 |
| $e^x$ | $e^x$ | 微分しても変わらない、変わり者 |
| $\log x$ | $1/x$ | 第5章の対数の傾き |
| $ax + b$ | $a$ | 直線の傾きは一定 |

例えばボンゴ焼きの損失 $L = \frac{1}{5}\sum (wx - y)^2$ を $w$ で微分すると、連鎖律で $\frac{1}{5}\sum 2(wx - y) \cdot x$ になります。$w = 1$ を入れると -44.04。自動微分の答えと一致します。

</details>

#### 📝 確認問題（`ch07-q14` · Lv.2 · 連鎖律 dy/dx = (dy/du)(du/dx) の手計算）

合成関数 $y = (2x)^3$ について、$u = 2x$ とおいて連鎖律 $\dfrac{dy}{dx} = \dfrac{dy}{du} \times \dfrac{du}{dx}$ を使い、$x = 1$ のときの $\dfrac{dy}{dx}$ を求めてください。（ヒント：$u^3$ の微分は $3u^2$）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：まず順伝播で $x = 1$ のときの $u = 2x$ の値を求め、次に外側の微分 $\frac{dy}{du}$ と内側の微分 $\frac{du}{dx}$ を掛け合わせます。
- **ヒント 2**：$u = 2$、$\frac{dy}{du} = 3u^2 = 3 \times 4$、$\frac{du}{dx} = 2$ です。これらを掛け算しましょう。
- **正解**：`24`

**考え方**：連鎖律は「順伝播で途中変数 $u$ の値を求め、逆向きに局所的な傾きを掛け算してつなぐ」伝言ゲームです。

1. 順伝播：$x = 1 \implies u = 2 \times 1 = 2 \implies y = 2^3 = 8$。
1. 外側の傾き：$\frac{dy}{du} = 3u^2 = 3 \times 2^2 = 12$。
1. 内側の傾き：$\frac{du}{dx} = 2$。
1. 掛け算：$\frac{dy}{dx} = 12 \times 2 = 24$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q14

</details>

#### 📝 確認問題（`ch07-q15` · Lv.4 · 【自力実装】y = (ax + b)^2 の手動逆伝播（連鎖律））

$u = a x + b$、$y = u^2$ という計算グラフについて、順伝播の値 `y` と、連鎖律で求めた3つの偏微分 `(dy_dx, dy_da, dy_db)` を返す関数 `forward_backward(a, b, x)` を純Pythonで自力実装し、数値微分との一致テストを通してください。

**ひな形コード：**
```python
def forward_backward(a, b, x):
    # 1. 順伝播: u = a * x + b, y = u ** 2
    # 2. 逆伝播: dy_du = 2 * u
    # 3. 連鎖律で dy_dx, dy_da, dy_db を計算して (y, dy_dx, dy_da, dy_db) を返す
    pass
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：まず順伝播で `u = a * x + b` と `y = u ** 2` を計算し、後ろから `dy_du = 2.0 * u` を求めます。
- **ヒント 2**：$u = ax + b$ の各偏微分（$x$ で $a$、$a$ で $x$、$b$ で $1$）それぞれに `dy_du` を掛けて返します。

**解答例：**
```python
def forward_backward(a, b, x):
    u = a * x + b
    y = u ** 2
    dy_du = 2.0 * u
    dy_dx = dy_du * a
    dy_da = dy_du * x
    dy_db = dy_du * 1.0
    return y, dy_dx, dy_da, dy_db
```

**検算テスト（Colabで実行して確認）：**
```python
y, dx, da, db = forward_backward(3.0, 1.0, 2.0)
assert abs(y - 49.0) < 1e-9
assert abs(dx - 42.0) < 1e-9
assert abs(da - 28.0) < 1e-9
assert abs(db - 14.0) < 1e-9
h = 1e-6
a, b, x = -1.5, 4.0, 2.5
y0, dx0, da0, db0 = forward_backward(a, b, x)
f = lambda aa, bb, xx: (aa * xx + bb) ** 2
assert abs(dx0 - (f(a, b, x + h) - f(a, b, x - h)) / (2 * h)) < 1e-5
assert abs(da0 - (f(a + h, b, x) - f(a - h, b, x)) / (2 * h)) < 1e-5
assert abs(db0 - (f(a, b + h, x) - f(a, b - h, x)) / (2 * h)) < 1e-5
print("ALL PASSED")
```

**考え方**：後ろから届いた勾配 `dy_du` を1回計算して使い回すだけで、入力 `x` とパラメータ `a, b` の勾配がすべて同時に求まります。これが誤差逆伝播法の本質です！

1. 順伝播：$u = ax + b$、$y = u^2$（`u` を覚えておくのがポイント）。
1. 逆伝播①：$\frac{dy}{du} = 2u$。
1. 逆伝播②：$\frac{\partial y}{\partial x} = (2u)\cdot a$、$\frac{\partial y}{\partial a} = (2u)\cdot x$、$\frac{\partial y}{\partial b} = (2u)\cdot 1$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html#ex-ch07-q15

</details>

*(この節の追加問題 `ch07-q16` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html で自動採点・復習できます)*

**✅ できたらチェック**

- [ ] 損失を $w$ の関数としてグラフにし、「谷」を見た
- [ ] 数値微分で傾きを測り、符号の意味を説明できる
- [ ] 勾配降下法で $w$ を自動で3付近まで動かせた
- [ ] 学習率を大きくしすぎると発散するのを見た
- [ ] `backward()` と `.grad` で自動微分を使えた
- [ ] 勾配を0に戻す理由（累積する）を説明できる

> **📝 まとめ**
>
> - **学習**とは、損失が小さくなるようにパラメータを調整すること
> - **微分**は傾き。傾きと逆向きに少しずつ進むのが **勾配降下法**：$w \leftarrow w - \text{lr} \times \frac{dL}{dw}$
> - パラメータが複数なら、偏微分を並べた **勾配** を使う
> - PyTorchの **自動微分** は、**連鎖律**で計算を逆にたどり、全パラメータの勾配を一度に求める（**誤差逆伝播法**）
> - 学習ループの基本形：① 損失計算 → ② `backward()` → ③ 更新（`no_grad`）→ ④ 勾配を0に

**ボンゴ先生**：道具は全部そろった。表（第6章）と、坂を下る方法（この章）。次の章では、この2つを合体させる。bigramの確率表の中身を「数える」のをやめて、勾配降下法で「学習」させるんだ。結果がどうなるか、予想しておいてくれ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html
- **次の章**：[第8章 数えずに学ぶ：ニューラルbigram](https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html)